# Stitching function 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import correlate, correlation_lags
from pathlib import Path
import napari

# ==========================================
# 1. THE STITCHER ENGINE (Profile-Based Consensus)
# ==========================================
def run_stitcher_test(vol1, vol2, axis=2, grid=(60, 20), expected=0, tolerance=200):
    """
    Finds shift by compressing 3D tiles into 1D profiles and 
    calculating a weighted consensus.
    """
    v1 = vol1.astype(np.float32)
    v2 = vol2.astype(np.float32)
    
    # Requirement from original script: ignore top 15 slices
    ignore_top = 15
    z_dim, y_dim, x_dim = v1.shape
    z_start, z_end = ignore_top, z_dim
    
    tile_z = (z_end - z_start) // grid[0]
    tile_y = y_dim // grid[1]
    
    all_shifts = []
    all_weights = []
    
    print(f"[Stitcher] Analyzing {grid[0]*grid[1]} tiles for consensus...")

    for r in range(grid[0]):
        for c in range(grid[1]):
            zs, ze = z_start + (r * tile_z), z_start + ((r + 1) * tile_z)
            ys, ye = c * tile_y, (c + 1) * tile_y
            
            # Extract Profiles (COMPRESSION STEP)
            prof1 = np.max(v1[zs:ze, ys:ye, :], axis=(0, 1))
            prof2 = np.max(v2[zs:ze, ys:ye, :], axis=(0, 1))

            # SNR Gate: Ignore dead tiles
            if np.std(prof1) < 1e-5 or np.max(prof1) < np.max(v1) * 0.1:
                continue

            # ZNCC Normalization
            p1_n = (prof1 - np.mean(prof1)) / (np.std(prof1) + 1e-10)
            p2_n = (prof2 - np.mean(prof2)) / (np.std(prof2) + 1e-10)
            
            corr = correlate(p1_n, p2_n, mode='full')
            lags = correlation_lags(len(p1_n), len(p2_n), mode='full')
            
            # Constraint mask
            mask = (lags >= expected - tolerance) & (lags <= expected + tolerance)
            if not np.any(mask): continue
            corr[~mask] = -np.inf 

            peak_idx = np.argmax(corr)
            weight = corr[peak_idx]
            
            all_shifts.append(lags[peak_idx])
            all_weights.append(weight)

    if not all_shifts:
        raise ValueError("Stitcher found no usable features!")

    # --- WEIGHTED CONSENSUS VOTE ---
    lag_min, lag_max = np.min(lags), np.max(lags)
    bins = np.arange(lag_min, lag_max + 2) - 0.5
    
    counts, bin_edges = np.histogram(all_shifts, bins=bins, weights=all_weights)
    best_bin = np.argmax(counts)
    final_shift = int(bin_edges[best_bin] + 0.5)
    
    print(f"[Stitcher] Consensus Result: {final_shift} px (Confidence: {np.max(all_weights):.3f})")
    return final_shift

# ==========================================
# 2. STANDALONE STITCHER EXECUTION
# ==========================================
if __name__ == "__main__":
    # --- CONFIG ---
    IN_DIR = Path.cwd().parent / 'DATA' / '2D TFM Data' / 'FeC Smile 3MHz 04022026 Filtered'
    
    # --- LOAD ---
    try:
        vol1 = np.load(IN_DIR / "FeC_40_5_filtered_3D_TFM.npy")
        vol2 = np.load(IN_DIR / "FeC_40_4_filtered_3D_TFM.npy")
    except FileNotFoundError:
        print("Data not found.")
        exit()

    # --- RUN ---
    stitch_shift = run_stitcher_test(vol1, vol2, grid=(60, 20))

    # --- VISUALIZE ---
    viewer = napari.Viewer(title="Stitcher Result Testing")
    clim = [0, np.percentile(vol1, 99.9)]
    
    viewer.add_image(vol1, name='Vol 1 (Reference)', colormap='cyan', contrast_limits=clim)
    
    trans = [0, 0, 0]
    trans[2] = stitch_shift
    viewer.add_image(vol2, name=f'Vol 2 (Stitcher Shift: {stitch_shift})', 
                     colormap='magenta', blending='additive', translate=trans, contrast_limits=clim)

    print(f"\nStitcher testing complete. View shift {stitch_shift} in Napari.")
    napari.run()

[Stitcher] Analyzing 1200 tiles for consensus...
[Stitcher] Consensus Result: 63 px (Confidence: 164.589)

Stitcher testing complete. View shift 63 in Napari.


: 

# Validation function

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import napari

class FeatureVoxelValidator:
    def __init__(self, vol1, vol2, axis=2, snr_threshold=0.1):
        """
        Validator that uses every pixel, but prioritizes features over noise.
        """
        # 1. Requirement: Crop first 10 pixels in Z
        v1_raw = vol1[10:, :, :].astype(np.float32)
        v2_raw = vol2[10:, :, :].astype(np.float32)
        
        # 2. FEATURE ENHANCEMENT (Similar to Stitcher's SNR Gate)
        # We find the global peak and ignore anything below a percentage of it
        # This prevents low-level noise from diluting the correlation score
        peak_val = np.max(v1_raw)
        thresh = peak_val * snr_threshold
        
        # Zero-out the noise background
        self.v1 = np.where(v1_raw > thresh, v1_raw, 0)
        self.v2 = np.where(v2_raw > thresh, v2_raw, 0)
        
        self.axis = axis
        self.shifts = []
        self.scores = []

    def validate_shift(self, test_range=range(-250, 251)):
        print(f"\n[Feature-Validator] Thresholding voxels < {np.max(self.v1)*0.1:.2f} (SNR Gate)")
        
        # Mean-centering ONLY the active voxels (ZNCC logic)
        v1_active = self.v1[self.v1 > 0]
        v1_mean = np.mean(v1_active) if v1_active.size > 0 else 0
        v1_centered = np.where(self.v1 > 0, self.v1 - v1_mean, 0)
        
        v1_flat = v1_centered.ravel()
        v1_norm = np.linalg.norm(v1_flat)
        
        self.shifts = list(test_range)
        self.scores = []

        for s in self.shifts:
            # Physical Translation
            shifted_v2 = np.roll(self.v2, shift=s, axis=self.axis)
            
            # Strict Masking (No wrap-around)
            mask = [slice(None)] * 3
            if s > 0:
                mask[self.axis] = slice(0, s)
                shifted_v2[tuple(mask)] = 0
            elif s < 0:
                mask[self.axis] = slice(s, None)
                shifted_v2[tuple(mask)] = 0
            
            # Feature-based normalization
            v2_active = shifted_v2[shifted_v2 > 0]
            v2_mean = np.mean(v2_active) if v2_active.size > 0 else 0
            v2_centered = np.where(shifted_v2 > 0, shifted_v2 - v2_mean, 0)
            
            v2_flat = v2_centered.ravel()
            v2_norm = np.linalg.norm(v2_flat)
            
            if v2_norm == 0:
                self.scores.append(0)
                continue
            
            # Full voxel dot product
            score = np.dot(v1_flat, v2_flat) / (v1_norm * v2_norm)
            self.scores.append(score)

        best_idx = np.argmax(self.scores)
        return self.shifts[best_idx], self.scores[best_idx]

# ==========================================
# EXECUTION
# ==========================================
if __name__ == "__main__":
    IN_DIR = Path.cwd().parent / 'DATA' / '2D TFM Data' / 'FeC Smile 3MHz 04022026 Filtered'
    vol1 = np.load(IN_DIR / "FeC_40_4_filtered_3D_TFM.npy")
    vol2 = np.load(IN_DIR / "FeC_40_3_filtered_3D_TFM.npy")

    # Run the Feature-Priority Validator
    validator = FeatureVoxelValidator(vol1, vol2, axis=2, snr_threshold=0.1)
    val_shift, val_score = validator.validate_shift()

    print(f"\n[Result] Feature-Matched Shift: {val_shift} px")
    print(f"[Result] Correlation Confidence: {val_score:.4f}")

    # Visual Inspection
    viewer = napari.Viewer()
    clim = [0, np.percentile(vol1, 99.9)]
    viewer.add_image(validator.v1, name='Vol1 (Thresholded)', colormap='cyan')
    
    trans = [0, 0, 0]; trans[2] = val_shift
    viewer.add_image(validator.v2, name=f'Vol2 (Shift {val_shift})', 
                     colormap='magenta', blending='additive', translate=trans)
    napari.run()


[Feature-Validator] Thresholding voxels < 1.61 (SNR Gate)

[Result] Feature-Matched Shift: 119 px
[Result] Correlation Confidence: 0.2261
